In [2]:
%pip install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install pyro-ppl seaborn

Note: you may need to restart the kernel to use updated packages.


In [4]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
import pandas as pd
import pyro
import pyro.distributions as dist
from pyro.infer import MCMC, NUTS
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from devinterp.optim.sgld import SGLD
from devinterp.optim.sgnht import SGNHT
from devinterp.slt import estimate_learning_coeff_with_summary, sample
from devinterp.slt.llc import LLCEstimator, SamplerCallback, OnlineLLCEstimator
from devinterp.utils import plot_trace

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
sns.set_style("whitegrid")

# plotting
CMAP = sns.color_palette("muted", as_cmap=True)
PRIMARY, SECONDARY, TERTIARY = CMAP[:3]

/var/folders/ds/qvg7h1s93x16vvczwp8vr83m0000gn/T/ipykernel_15874/2903749014.py:4: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd
/Users/ragharao/opt/anaconda3/envs/devinterp_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# constants
SIGMA = 0.25
NUM_TRAIN_SAMPLES = 1000
BATCH_SIZE = NUM_TRAIN_SAMPLES
CRITERION = F.mse_loss

In [6]:
class SingularModel(nn.Module):
    def __init__(self, powers):
        super(SingularModel, self).__init__()
        self.weights = nn.Parameter(
            torch.tensor([0.0, 0.0], dtype=torch.float32, requires_grad=True).to(DEVICE)
        )
        self.powers = powers

    def forward(self, x):
        multiplied = torch.prod(self.weights**self.powers)
        x = x * multiplied
        return x

def generate_dataset_for_seed(seed=0):
    torch.manual_seed(seed)
    np.random.seed(seed)
    x = torch.normal(0, 2, size=(NUM_TRAIN_SAMPLES,))
    y = SIGMA * torch.normal(0, 1, size=(NUM_TRAIN_SAMPLES,))
    train_data = TensorDataset(x, y)
    train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
    return train_loader, train_data, x, y

In [7]:
train_loader, train_data, x, y = generate_dataset_for_seed(0)
lr = 0.0001
num_chains = 3
num_draws = 5000
powers_to_test = [
    torch.tensor([0, 2]).to(DEVICE), # minimally singular
    torch.tensor([2, 0]).to(DEVICE) # minimally sigular
]
rlct_estimates_sgld = {
    tuple(power.detach().cpu().tolist()): estimate_learning_coeff_with_summary(
        model=SingularModel(power),
        loader=train_loader,
        optimizer_kwargs=dict(
            lr=lr,
            bounding_box_size=1.0,
        ),
        criterion=CRITERION,
        sampling_method=SGLD,
        num_chains=num_chains,
        num_draws=num_draws,
        device=DEVICE,
    )
    for power in powers_to_test
}
rlct_estimates_sgnht = {
    tuple(power.detach().cpu().tolist()): estimate_learning_coeff_with_summary(
        model=SingularModel(power),
        loader=train_loader,
        optimizer_kwargs=dict(
            lr=lr,
            bounding_box_size=1.0,
            diffusion_factor=0.01,
        ),
        criterion=CRITERION,
        sampling_method=SGNHT,
        num_chains=num_chains,
        num_draws=num_draws,
        device=DEVICE,
    )
    for power in powers_to_test
}

/Users/ragharao/opt/anaconda3/envs/devinterp_env/lib/python3.11/site-packages/devinterp/slt/sampler.py:170: UserWarning: You are taking more draws than burn-in steps, your LLC estimates will likely be underestimates. Please check LLC chain convergence.
  warnings.warn(
/Users/ragharao/opt/anaconda3/envs/devinterp_env/lib/python3.11/site-packages/devinterp/slt/sampler.py:174: UserWarning: You are taking more sample batches than there are dataloader batches available, this removes some randomness from sampling but is probably fine. (All sample batches beyond the number dataloader batches are cycled from the start, f.e. 9 samples from [A, B, C] would be [B, A, C, B, A, C, B, A, C].)
  warnings.warn(
/Users/ragharao/opt/anaconda3/envs/devinterp_env/lib/python3.11/site-packages/devinterp/slt/sampler.py:58: UserWarning: You are taking more sample batches than there are dataloader batches available, this removes some randomness from sampling but is probably fine. (All sample batches beyond th

In [8]:
for sampler_type, estimates in zip(
    ("sgld", "sgnht"), (rlct_estimates_sgld, rlct_estimates_sgnht)
):
    df_data = {
        "powers": [k for k in estimates.keys()],
        "LLC_Summary": estimates.values(),
    }
    df = pd.DataFrame(df_data)
    df["llc_mean"] = df["LLC_Summary"].apply(lambda x: x["llc/mean"])
    df["llc_std"] = df["LLC_Summary"].apply(lambda x: x["llc/std"])
    df = df.drop("LLC_Summary", axis=1)
    print(sampler_type)
    print(df.to_string(index=False))

sgld
powers  llc_mean  llc_std
(0, 2)  0.235147 0.050897
(2, 0)  0.177827 0.032972
sgnht
powers  llc_mean  llc_std
(0, 2)  0.279457 0.024806
(2, 0)  0.281580 0.024980


In [16]:
def sum(n):
    if n == 1:
        return 1
    return n + sum(n-1)

for i in range(1000, 2000):
    print(i)

1000
1001
1002
1003
1004
1005
1006
1007
1008
1009
1010
1011
1012
1013
1014
1015
1016
1017
1018
1019
1020
1021
1022
1023
1024
1025
1026
1027
1028
1029
1030
1031
1032
1033
1034
1035
1036
1037
1038
1039
1040
1041
1042
1043
1044
1045
1046
1047
1048
1049
1050
1051
1052
1053
1054
1055
1056
1057
1058
1059
1060
1061
1062
1063
1064
1065
1066
1067
1068
1069
1070
1071
1072
1073
1074
1075
1076
1077
1078
1079
1080
1081
1082
1083
1084
1085
1086
1087
1088
1089
1090
1091
1092
1093
1094
1095
1096
1097
1098
1099
1100
1101
1102
1103
1104
1105
1106
1107
1108
1109
1110
1111
1112
1113
1114
1115
1116
1117
1118
1119
1120
1121
1122
1123
1124
1125
1126
1127
1128
1129
1130
1131
1132
1133
1134
1135
1136
1137
1138
1139
1140
1141
1142
1143
1144
1145
1146
1147
1148
1149
1150
1151
1152
1153
1154
1155
1156
1157
1158
1159
1160
1161
1162
1163
1164
1165
1166
1167
1168
1169
1170
1171
1172
1173
1174
1175
1176
1177
1178
1179
1180
1181
1182
1183
1184
1185
1186
1187
1188
1189
1190
1191
1192
1193
1194
1195
1196
1197
1198
1199


In [ ]:
hessians = produce_hessians(models=models,
                                data_loader=train_loader,
                                num_batches=args.hessian_batch_size,
                                criterion=metric,
                                device=device,
                                history=True)